In [ ]:
import pandas as pd
df = pd.read_excel("WikiRelationsToVerify.xlsx", sheet_name="Relations with limited PMI")

In [ ]:
import requests
import pandas as pd

def fetch_wikidata_labels(ids):
    """Fetches labels for a list of Wikidata QIDs or PIDs in batches."""
    unique_ids = list(set(ids))
    id_to_label = {}
    batch_size = 500

    for i in range(0, len(unique_ids), batch_size):
        batch = unique_ids[i:i+batch_size]
        values_str = " ".join(f"wd:{item}" for item in batch if pd.notna(item) and str(item).startswith(('Q', 'P')))
        if not values_str:
            continue

        query = f"""
        SELECT ?item ?itemLabel WHERE {{
          VALUES ?item {{ {values_str} }}
          SERVICE wikibase:label {{ bd:serviceParam wikibase:language "[AUTO_LANGUAGE],en". }}
        }}
        """

        url = 'https://query.wikidata.org/sparql'
        headers = {
            'User-Agent': 'ColabNotebook/1.0 (contact@example.com)',
            'Accept': 'application/json'
        }

        try:
            response = requests.get(url, params={'query': query, 'format': 'json'}, headers=headers)
            if response.status_code == 200:
                results = response.json().get('results', {}).get('bindings', [])
                for row in results:
                    item_uri = row.get('item', {}).get('value', '')
                    item_id = item_uri.split('/')[-1]
                    label = row.get('itemLabel', {}).get('value', '')
                    id_to_label[item_id] = label
            else:
                print(f"Failed batch {i // batch_size + 1}: Status {response.status_code}")
        except Exception as e:
            print(f"Error fetching batch: {e}")

    return id_to_label

# Gather all unique QIDs/PIDs that need to be resolved
all_ids = pd.concat([df['subject'], df['object'], df['prop']]).dropna().unique()

# Fetch labels from Wikidata
print(f"Fetching labels for {len(all_ids)} unique IDs...")
labels_map = fetch_wikidata_labels(all_ids)

# Map labels back to the dataframe
df['subject_label'] = df['subject'].map(labels_map)
df['object_label'] = df['object'].map(labels_map)
df['prop_label'] = df['prop'].map(labels_map)

# Display a sample of the resulting DataFrame
df.head()

Fetching labels for 6806 unique IDs...


,subject,subjectID,object,objectID,prop,PMI,subject_label,object_label,prop_label
0,Q58833983,D064726,Q12136,D004194,P31,-5.731662,triple-negative breast neoplasm,disease,instance of
1,Q4082046,D063146,Q623,D002244,P527,-5.628619,polycomb-group proteins,carbon,has part(s)
2,Q382370,D045908,Q835153,D010808,P923,-5.225243,mpox,physical examination,medical examination
3,Q155098,D007922,Q166774,D011480,P924,-5.046738,leptospirosis,protease inhibitor,possible treatment
4,Q1059765,D010510,Q120569,D005762,P1995,-4.978983,periodontal disease,gastroenterology,health specialty


In [ ]:
!pip install typesafe-sdk

In [ ]:
!uv add typesafe-sdk

error: No `pyproject.toml` found in current directory or any parent directory


In [ ]:
import os
os.environ["TYPESAFE_API_KEY"] = "YOUR_API_KEY" # Replace with your actual API key

In [ ]:
from typesafe_sdk import Choice, TypeSafeClient
from tqdm import tqdm

findings = []
with TypeSafeClient() as client:
  for rel in tqdm(df.iloc):
    inst=f"Determine whether the semantic relation where {rel["subject_label"]} is the subject, {rel["prop_label"]} is the predicate, and {rel["object_label"]} is the object constitutes an accurate semantic relation."
    print(inst)
    response = client.system_one(
        state={"document": "You are evaluating candidate relations extracted from a knowledge graph. The goal is to determine whether the subject-predicate–object triple represents a valid and meaningful relationship according to established scientific knowledge and terminology."},
        questions={
            "category": Choice(
                instructions=inst,
                criteria={"true": None, "maybe": None, "false": None},
            ),
        },
    )
    findings.append(response.choices["category"].choice)
    print(response.choices["category"].choice)

0it [00:00, ?it/s]

Determine whether the semantic relation where triple-negative breast neoplasm is the subject, instance of is the predicate, and disease is the object constitutes an accurate semantic relation.


2it [00:00,  4.63it/s]

true
Determine whether the semantic relation where polycomb-group proteins is the subject, has part(s) is the predicate, and carbon is the object constitutes an accurate semantic relation.
false
Determine whether the semantic relation where mpox is the subject, medical examination is the predicate, and physical examination is the object constitutes an accurate semantic relation.


4it [00:00,  4.97it/s]

...

true
Determine whether the semantic relation where lead poisoning of the nervous system is the subject, health specialty is the predicate, and neurology is the object constitutes an accurate semantic relation.
true
Determine whether the semantic relation where scoliosis is the subject, subclass of is the predicate, and disease is the object constitutes an accurate semantic relation.


15923it [53:56,  5.15it/s]

true
Determine whether the semantic relation where inguinal hernia is the subject, subclass of is the predicate, and intestinal disease is the object constitutes an accurate semantic relation.


15925it [53:57,  4.92it/s]

false
Determine whether the semantic relation where transformation is the subject, different from is the predicate, and genetics is the object constitutes an accurate semantic relation.
false


In [ ]:
df["choice"] = findings

In [ ]:
df['choice'].value_counts()

,count
choice,
true,10407
false,5355
maybe,163


In [ ]:
df.to_excel("Low PMI.xlsx", index=False)